In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# nolds is not in Colab's default environment
!pip install nolds --quiet

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
files.upload()

Saving compute_omega_all.py to compute_omega_all.py


{'compute_omega_all.py': b'"""\ncompute_omega_all.py\n====================\nDay 3 task: compute spectral predictability (Omega) + ACF@24h + CV + Hurst\nfor Alibaba and Bitbrains datasets, then print the comparison table.\n\nByteDance IaaS is already computed (hardcoded at the bottom).\n\nRun on Vast.ai:\n    python3 compute_omega_all.py --mode alibaba  (or bitbrains, or both)\n\nOutputs (written to OUTPUT_DIR):\n    omega_alibaba.csv   - per-container metrics\n    omega_bitbrains.csv - per-VM metrics\n    omega_summary.csv   - the final 3-row comparison table for the thesis\n\nCheckpoints are saved so you can resume if the run dies mid-way.\nAlibaba (~4900 containers) takes ~10-15 min.\nBitbrains (~1200 VMs after filter) takes ~5 min.\n\nAuthor: Jimmy (ITDSIU21078)\n"""\n\nimport os\nimport json\nimport time\nimport argparse\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import periodogram\n\nwarnings.filterwarnings(\'ignore\')\n\n\n# \xe2\x94\x80\xe2\x9

In [ ]:
# run this first to check, don't touch the script yet
import os

ALIBABA_PARQUET = '/content/drive/MyDrive/workspace/thesis/train.parquet'
BITBRAINS_DIR   = '/content/drive/MyDrive/Thesis/bitbrains/fastStorage/2013-8'

print('Alibaba parquet exists:', os.path.exists(ALIBABA_PARQUET))

csvs = [f for f in os.listdir(BITBRAINS_DIR) if f.endswith('.csv')]
print(f'Bitbrains CSVs found: {len(csvs)}')
print(f'sample file: {csvs[0]}')  # should print something like 994.csv

Alibaba parquet exists: True
Bitbrains CSVs found: 1250
sample file: 137.csv


In [ ]:
%run compute_omega_all.py --mode both

[OK] nolds found and warmed up - Hurst will be computed
ALIBABA
loading /content/drive/MyDrive/workspace/thesis/train.parquet ...
loaded 7,413,073 rows
columns: ['time_stamp', 'container_id', 'machine_id', 'cpu_util_percent', 'mem_util_percent', 'disk_io_percent', 'net_in', 'net_out', 'cluster_id', 'workload_cluster']
using container col: 'container_id', cpu col: 'cpu_util_percent'
found 4902 containers
  [200/4902] 4.1% done  elapsed 123s
  [400/4902] 8.2% done  elapsed 245s
  [600/4902] 12.2% done  elapsed 368s
  [800/4902] 16.3% done  elapsed 491s
  [1000/4902] 20.4% done  elapsed 614s
  [1200/4902] 24.5% done  elapsed 736s
  [1400/4902] 28.6% done  elapsed 858s
  [1600/4902] 32.6% done  elapsed 980s
  [1800/4902] 36.7% done  elapsed 1102s
  [2000/4902] 40.8% done  elapsed 1224s
  [2200/4902] 44.9% done  elapsed 1346s
  [2400/4902] 49.0% done  elapsed 1468s
  [2600/4902] 53.0% done  elapsed 1590s
  [2800/4902] 57.1% done  elapsed 1713s
  [3000/4902] 61.2% done  elapsed 1836s
  [3200

In [ ]:
import pandas as pd

# per-container results
ali = pd.read_csv('./omega_results/omega_alibaba.csv')
print(f'Alibaba: {len(ali)} containers')
print(ali[['cv', 'hurst', 'acf_24h', 'omega']].describe().round(3))

bb = pd.read_csv('./omega_results/omega_bitbrains.csv')
print(f'\nBitbrains: {len(bb)} VMs')
print(bb[['cv', 'hurst', 'acf_24h', 'omega']].describe().round(3))

# the final thesis table
summary = pd.read_csv('./omega_results/omega_summary.csv')
print('\n--- THESIS TABLE ---')
print(summary.to_string(index=False))

Alibaba: 4902 containers
             cv     hurst   acf_24h     omega
count  4686.000  4684.000  4400.000  4684.000
mean      0.859     0.784     0.377     0.385
std       3.105     0.098     0.339     0.187
min       0.000     0.409    -0.273     0.000
25%       0.288     0.710     0.049     0.213
50%       0.373     0.781     0.316     0.346
75%       0.498     0.872     0.718     0.564
max      40.137     1.093     0.981     0.901

Bitbrains: 156 VMs
            cv    hurst  acf_24h    omega
count  156.000  156.000  156.000  156.000
mean     1.043    0.991    0.231    0.477
std      0.313    0.076    0.239    0.107
min      0.095    0.420   -0.062    0.313
25%      0.957    0.955    0.047    0.409
50%      1.056    0.997    0.116    0.436
75%      1.129    1.030    0.444    0.582
max      1.850    1.333    0.921    0.848

--- THESIS TABLE ---
       Dataset  N_series  CV_median  Hurst_median  ACF@1h  ACF@24h  Omega_median
     Bitbrains       156      1.056         0.997   0.748   

In [ ]:
import shutil, os

os.makedirs('/content/drive/MyDrive/workspace/thesis/omega_results', exist_ok=True)

for fname in ['omega_alibaba.csv', 'omega_bitbrains.csv', 'omega_summary.csv']:
    shutil.copy(
        f'./omega_results/{fname}',
        f'/content/drive/MyDrive/workspace/thesis/omega_results/{fname}'
    )
    print(f'copied {fname}')

copied omega_alibaba.csv
copied omega_bitbrains.csv
copied omega_summary.csv
